<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Cristian Jara
- Nombre de alumno 2: Felipe Fierro


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/cristianjaram/MDS7202-labs/tree/main/labs)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [1]:
!pip install -qq xgboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 7.9 MB/s eta 0:00:00


# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.



In [1]:
import pandas as pd
import numpy as np
from datetime import datetime


df = pd.read_csv("sales.csv")

df.head()

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


In [2]:
df.tail()

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
7451,7555,31/12/18,Athens,37.97945,23.71622,664046,shop_1,kinder-cola,plastic,1.5lt,2.52,13760
7452,7556,31/12/18,Athens,37.97945,23.71622,664046,shop_1,orange-power,plastic,1.5lt,2.18,16309
7453,7557,31/12/18,Patra,38.24444,21.73444,168034,shop_6,kinder-cola,can,330ml,0.85,24378
7454,7558,31/12/18,Thessaloniki,40.64361,22.93086,354290,shop_4,adult-cola,plastic,1.5lt,2.17,20691
7455,7559,31/12/18,Irakleion,35.32787,25.14341,137154,shop_2,adult-cola,glass,500ml,1.26,24615


In [ ]:
!pip install ydata_profiling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.3/399.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2


In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Reporte del Dataset", explorative=True)
profile.to_file("reporte.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [00:00<00:00, 32.49it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## 1 Generando un Baseline (5 puntos)

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [3]:
from sklearn import set_config
set_config(transform_output="pandas")

# Inserte su código acá


In [4]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import joblib

In [5]:
# Definimos target
target = "quantity"
X = df.drop(columns=[target, "id"])
y = df[target]


In [6]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.222, random_state=42  # 0.222 ~ 20% total
)

In [29]:
# 3. Feature engineering para 'date'
# =====================
def extract_date_parts(df):
    df = df.copy()
    df["year"] = pd.to_datetime(df["date"]).dt.year.astype("category")
    df["month"] = pd.to_datetime(df["date"]).dt.month.astype("category")
    df["day"] = pd.to_datetime(df["date"]).dt.day.astype("category")
    df.drop(columns=["date"])
    # Limpieza de capacity: extrae solo números (por ejemplo "330ml" -> 330)
    if "capacity" in df.columns:
        df["capacity"] = (
            df["capacity"]
            .astype(str)
            .apply(lambda x: float(re.findall(r"\d+\.?\d*", x)[0]) if re.findall(r"\d+\.?\d*", x) else np.nan)
        )

    return df

date_transformer = FunctionTransformer(extract_date_parts)


In [60]:
# 4. ColumnTransformer
# =====================
num_features = ["lat", "long", "pop", "capacity", "price"]
cat_features = ["city", "shop", "brand", "container", "year", "month", "day"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False
), cat_features)
    ]
).set_output(transform="pandas")


In [34]:

dummy_pipeline = Pipeline([
    ("date", date_transformer),
    ("preprocess", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

dummy_pipeline.fit(X_train, y_train)
y_val_pred = dummy_pipeline.predict(X_val)
mae_dummy = mean_absolute_error(y_val, y_val_pred)

print(f"MAE DummyRegressor: {mae_dummy:.2f}")


MAE DummyRegressor: 13545.19


In [35]:
# 6. Pipeline con XGBRegressor (default)
# =====================
xgb_pipeline = Pipeline([
    ("date", date_transformer),
    ("preprocess", preprocessor),
    ("model", XGBRegressor(random_state=42, verbosity=0))
])

xgb_pipeline.fit(X_train, y_train)
y_val_pred = xgb_pipeline.predict(X_val)
mae_xgb = mean_absolute_error(y_val, y_val_pred)
print(f"MAE XGBRegressor: {mae_xgb:.2f}")

MAE XGBRegressor: 2409.31


In [20]:
# 7. Guardar modelos
# =====================
joblib.dump(dummy_pipeline, "dummy.pkl")
joblib.dump(xgb_pipeline, "xgb.pkl")

['xgb.pkl']

El modelo base (DummyRegressor) alcanzó un error absoluto medio (MAE) de 13.545 unidades, lo que implica que, en promedio, las predicciones difieren en más de 13 mil unidades respecto a la demanda real. Este desempeño es esperable, dado que el modelo solo predice el promedio histórico sin aprovechar información contextual. En contraste, el modelo XGBRegressor, utilizando sus parámetros por defecto, redujo el MAE a 2.409 unidades, evidenciando una mejora significativa en la capacidad predictiva. Esto demuestra que el algoritmo logra capturar patrones en variables como ciudad, marca, precio, capacidad y estacionalidad (mes o año), disminuyendo el error promedio respecto al modelo base. En términos de negocio, esta mejora representa una reducción sustancial en la incertidumbre de demanda, permitiendo planificación más precisa de inventarios y producción, con menor riesgo de sobrestock o quiebres.

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)



Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [38]:

feature_names = preprocessor.get_feature_names_out()
price_index = np.where(feature_names == "num__price")[0][0]
print(f"Índice de 'price' en las features: {price_index}")

Índice de 'price' en las features: 4


In [39]:

X_train_transformed = date_transformer.transform(X_train)
X_train_preprocessed = preprocessor.fit_transform(X_train_transformed)


constraints = [0] * len(feature_names)
constraints[4] = -1  # Forzamos relación negativa para 'price'


xgb_monotone = XGBRegressor(
    random_state=42,
    verbosity=0,
    monotone_constraints=tuple(constraints)
)


xgb_monotone_pipeline = Pipeline([
    ("date", date_transformer),
    ("preprocess", preprocessor),
    ("model", xgb_monotone)
])

xgb_monotone_pipeline.fit(X_train, y_train)
y_val_pred_monotone = xgb_monotone_pipeline.predict(X_val)
mae_monotone = mean_absolute_error(y_val, y_val_pred_monotone)
print(f"MAE XGBRegressor (monotone constraint): {mae_monotone:.2f}")


joblib.dump(xgb_monotone_pipeline, "xgb_monotone.pkl")


MAE XGBRegressor (monotone constraint): 2555.88


['xgb_monotone.pkl']

Al imponer una relación inversa entre el precio y la demanda, el MAE pasó de 2.409 a 2.555.
Aunque el error aumentó ligeramente, el modelo ahora respeta una lógica económica coherente, lo que lo hace más confiable para decisiones de pricing. Por lo que en ese sentido, el amigo si tiene razón para poder tener un modelo muchísimo más realista en cuanto a la teoría de precio-demanda.

## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)


Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [40]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.metrics import mean_absolute_error
import time


def objective(trial):

    min_freq = trial.suggest_float("ohe_min_freq", 0.0, 1.0)


    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "max_leaves": trial.suggest_int("max_leaves", 0, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 1),
        "random_state": 42,
        "verbosity": 0,
    }


    preprocessor_opt = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=min_freq),
             cat_features),
        ]
    ).set_output(transform="pandas")


    model = XGBRegressor(**params)

    pipeline = Pipeline([
        ("date", date_transformer),
        ("preprocess", preprocessor_opt),
        ("model", model)
    ])


    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)


    trial.set_user_attr("pipeline", pipeline)

    return mae

In [41]:

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

start_time = time.time()
study.optimize(objective, timeout=300)  # 5 minutos = 300 segundos
elapsed = time.time() - start_time

In [42]:
print(f"Tiempo total de optimización: {elapsed/60:.2f} minutos")
print(f"Trials completados: {len(study.trials)}")
print(f"Mejor MAE: {study.best_value:.4f}")
print("Mejores hiperparámetros encontrados:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Tiempo total de optimización: 5.03 minutos
Trials completados: 173
Mejor MAE: 1955.1337
Mejores hiperparámetros encontrados:
  ohe_min_freq: 0.0005059112090852677
  learning_rate: 0.09242832745715192
  n_estimators: 879
  max_depth: 10
  max_leaves: 57
  min_child_weight: 3
  reg_alpha: 0.9701431937240514
  reg_lambda: 0.8574212979481073


In [43]:

best_pipeline = study.best_trial.user_attrs["pipeline"]
joblib.dump(best_pipeline, "xgb_optuna.pkl")

['xgb_optuna.pkl']

Al aplicar la optimización bayesiana con Optuna (TPESampler), el error absoluto medio (MAE) del modelo disminuyó desde aproximadamente 2.409 (modelo XGBoost por defecto) hasta 1.955, lo que representa una mejora cercana al 19%.

Esta mejora se debe principalmente a que el proceso de optimización:

- Ajustó de forma automática los hiperparámetros más sensibles del modelo (profundidad, tasa de aprendizaje, regularizaciones, etc.), encontrando una combinación más equilibrada entre sesgo y varianza.

- Exploró el espacio de parámetros de forma inteligente y dirigida, priorizando las zonas del espacio donde las pruebas anteriores mostraron mejor desempeño (ventaja del método bayesiano frente a una búsqueda aleatoria o de cuadrícula).

- Ajustó también el parámetro min_frequency del OneHotEncoder, lo que ayudó a reducir ruido de categorías raras sin eliminar información relevante.

En cuanto a los parámetros y sus rangos

- learning_rate: controla qué tan rápido aprende el modelo en cada iteración; valores bajos dan más precisión, altos más velocidad. Rango (0.001–0.1) adecuado.

- n_estimators: número de árboles; más árboles aumentan precisión pero también riesgo de sobreajuste. El Rango (50–1000) es razonable.

- max_depth: profundidad de los árboles; mayor profundidad permite relaciones más complejas, pero puede sobreajustar. El Rango (3–10) es correcto.

- max_leaves: número máximo de hojas por árbol; limita la complejidad interna. Rango (0–100) amplio y apropiado.

- min_child_weight: mínimo de observaciones por hoja; evita ramas por ruido. El Rango (1–5) adecuado.

- reg_alpha: regularización L1; elimina variables poco relevantes. Por lo que el Rango (0–1) razonable.

- reg_lambda: regularización L2; reduce varianza del modelo. El Rango (0–1) correcto.

- min_frequency (OneHotEncoder): controla qué tan frecuentes deben ser las categorías para ser codificadas es decir evita columnas por categorías raras. El Rango (0.0–1.0) adecuado.

## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)



Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [101]:
import optuna
from optuna.samplers import TPESampler
from optuna.exceptions import TrialPruned
from optuna.integration import XGBoostPruningCallback
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
import re

In [44]:
!pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 2.7 MB/s eta 0:00:00


In [90]:
def extract_date_parts(df):
    df = df.copy()
    if "date" not in df.columns:
        return df

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["year"] = df["date"].dt.year.astype("category")
    df["month"] = df["date"].dt.month.astype("category")
    df["day"] = df["date"].dt.day.astype("category")
    df = df.drop(columns=["date"], errors="ignore")

    if "capacity" in df.columns:
        df["capacity"] = (
            df["capacity"]
            .astype(str)
            .apply(lambda x: float(re.findall(r"\d+\.?\d*", x)[0]) if re.findall(r"\d+\.?\d*", x) else np.nan)
        )

    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype("category")

    if "pop" in df.columns:
        df["pop"] = df["pop"].astype(float)

    return df



In [91]:
date_transformer = FunctionTransformer(extract_date_parts, validate=False)

In [102]:
def objective_pruning(trial):
    global X_train, X_val, y_train, y_val

    min_freq = trial.suggest_float("ohe_min_freq", 0.0, 1.0)

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "max_leaves": trial.suggest_int("max_leaves", 0, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 1),
        "random_state": 42,
        "verbosity": 0,
        "eval_metric": "mae"
    }

    preprocessor_opt = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore",
                                  sparse_output=False,
                                  min_frequency=min_freq),
             cat_features),
        ]
    ).set_output(transform="pandas")

    pipeline = Pipeline([
        ("date", date_transformer),
        ("preprocess", preprocessor_opt),
        ("model", XGBRegressor(**params))
    ])

    pipeline.fit(X_train, y_train, model__verbose=False)
    y_pred = pipeline.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    trial.report(mae, step=0)
    if trial.should_prune():
        raise TrialPruned()

    trial.set_user_attr("pipeline", pipeline)
    return mae


In [103]:
sampler = TPESampler(seed=42)
study_prune = optuna.create_study(direction="minimize", sampler=sampler)
study_prune.optimize(objective_pruning, timeout=300, show_progress_bar=True)

   0%|          | 00:00/05:00

In [104]:
print(f"Trials completados: {len(study_prune.trials)}")
print(f"Mejor MAE: {study_prune.best_value:.4f}")
print("Mejores hiperparámetros encontrados:")
for k, v in study_prune.best_params.items():
    print(f"  {k}: {v}")




Trials completados: 145
Mejor MAE: 1956.9753
Mejores hiperparámetros encontrados:
  ohe_min_freq: 0.021894054058511806
  learning_rate: 0.08430204550474919
  n_estimators: 828
  max_depth: 10
  max_leaves: 68
  min_child_weight: 3
  reg_alpha: 0.9629617749790562
  reg_lambda: 0.9563206926287683


In [105]:
best_pipeline_prune = study_prune.best_trial.user_attrs["pipeline"]
joblib.dump(best_pipeline_prune, "xgb_optuna_pruned.pkl")

['xgb_optuna_pruned.pkl']

- Lo que hace prunning, es permitir detener tempranamente los trials que muestran un mal desempeño parcial. Esto al fin y al cabo reduce tiempo total y permite explorar más configuraciones en el mismo plazo.

Su objetivo es ahorrar tiempo y recursos, enfocando la búsqueda en combinaciones prometedoras. En teoría, debería mantener un rendimiento similar pero reducir el tiempo total de entrenamiento.

En cuanto a los resultados: El MAE se mantuvo prácticamente igual, pero el número de trials completados fue menor. Esto indica que el pruning mejoró la eficiencia sin afectar la precisión. La ligera diferencia en el error se debe a la naturaleza aleatoria del muestreo bayesiano y a que algunos trials se interrumpen antes de finalizar.

## 5. Visualizaciones (5 puntos)


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [107]:
# Inserte su código acá
import optuna
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_param_importances
import plotly.io as pio

pio.renderers.default = "colab"

# 1. Gráfico de historial de optimización
fig_hist = plot_optimization_history(study)
fig_hist.show()

# 2. Gráfico de coordenadas paralelas
fig_para = plot_parallel_coordinate(study)
fig_para.show()

# 3. Gráfico de importancia de hiperparámetros
fig_imp = plot_param_importances(study)
fig_imp.show()


Las mejoras notables en el MAE aparecen a partir del trial 5 aproximadamente.
En los primeros intentos, el modelo presentaba errores elevados (MAE > 6000–8000), pero tras los primeros cinco trials, el valor objetivo cae bruscamente y se estabiliza alrededor de 2000, manteniendo variaciones menores.
Esto nos dice que el optimizador encontró rápidamente una región prometedora del espacio de hiperparámetros y luego refinó ajustes dentro de ella.

Del gráfico de coordenadas paralelas se observan tres patrones principales:

- Learning rate alto (≈0.08–0.1) y mayor número de árboles (n_estimators entre 800–1000) tienden a correlacionarse con los mejores resultados por líneas más oscuras, lo que implica MAE más bajo.

- Profundidad máxima (max_depth = 10) se repite en los mejores trials, lo que indica que una mayor capacidad del modelo fue beneficiosa en este caso.

- Los valores de regularización (reg_alpha y reg_lambda) cercanos a 1 aparecen con frecuencia en los mejores resultados, mostrando que una alta regularización contribuye a estabilidad del modelo.


## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [111]:
# Inserte su código acá

import joblib
from sklearn.metrics import mean_absolute_error

# Cargar modelos
models = {
    "Dummy": joblib.load("dummy.pkl"),
    "XGB Default": joblib.load("xgb.pkl"),
    "XGB Monotone": joblib.load("xgb_monotone.pkl"),
    "XGB Optuna": joblib.load("xgb_optuna.pkl"),
    "XGB Optuna + Pruning": joblib.load("xgb_optuna_pruned.pkl"),
}

# Evaluar MAE en validación
mae_results = {}
for name, model in models.items():
    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mae_results[name] = mae

# Mostrar tabla resumen
import pandas as pd
mae_table = pd.DataFrame(mae_results.items(), columns=["Modelo", "MAE"]).sort_values(by="MAE")
display(mae_table)


,Modelo,MAE
3,XGB Optuna,1955.133667
4,XGB Optuna + Pruning,1956.975342
1,XGB Default,2409.308105
2,XGB Monotone,2555.875244
0,Dummy,13545.190961


El modelo XGB Optuna obtuvo el mejor rendimiento, con un MAE de 1955.13 en validación.
La búsqueda bayesiana de hiperparámetros permitió encontrar una combinación más equilibrada entre complejidad y regularización, superando al modelo por defecto, al monotónico y al de pruning.

In [112]:
best_model = joblib.load("xgb_optuna.pkl")
y_test_pred = best_model.predict(X_test)
mae_test = mean_absolute_error(y_test, y_test_pred)

print(f"MAE en conjunto de test: {mae_test:.2f}")


MAE en conjunto de test: 1862.29


Sí, el MAE del conjunto de test fue 1862.29, ligeramente menor al obtenido en validación (1955.13).
Esta diferencia se debe a la variabilidad natural de los datos y al hecho de que el conjunto de test contiene observaciones nuevas no vistas durante el entrenamiento.
Como la diferencia es pequeña, indica que el modelo generaliza bien y no sobreajusta, manteniendo un desempeño consistente en datos no utilizados durante la optimización.

# Conclusión
Exito!
